# Feature engineering

I build pre-match features for each team. Every feature uses only matches played before the one I predict.

## 1. Recent form
For each team and each match, I compute the average points over the 5 previous matches and the goals scored and conceded over the 10 previous matches.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.database import query

In [5]:
query("""
    SELECT m.date, m.opponent, m.goals_for, m.goals_against,
           f.n_previous, f.points_last_5, f.goals_for_last_10, f.goals_against_last_10
    FROM team_matches AS m
    JOIN team_form AS f USING (match_id, team)
    WHERE m.team = 'France'
    ORDER BY m.date DESC
    LIMIT 8
""")

,date,opponent,goals_for,goals_against,n_previous,points_last_5,goals_for_last_10,goals_against_last_10
0,2026-07-18,England,4,6,939,2.4,2.3,0.8
1,2026-07-14,Spain,0,2,938,3.0,2.5,0.7
2,2026-07-09,Morocco,2,0,937,3.0,2.6,0.8
3,2026-07-04,Paraguay,1,0,936,3.0,2.9,0.8
4,2026-06-30,Sweden,3,0,935,2.4,2.8,1.0
5,2026-06-26,Norway,4,1,934,2.4,2.7,0.9
6,2026-06-22,Iraq,3,0,933,2.4,2.6,1.0
7,2026-06-16,Senegal,3,1,932,2.4,2.5,0.9


Before the semi-final against Spain, France had won its 5 previous matches, so `points_last_5` is 3.0.
The loss against Spain is not in its own row. It only enters the next row: (0 + 3 + 3 + 3 + 3) / 5 = 2.4. The window stops before the current match, so there is no leakage.

In [3]:
query("""
    SELECT COUNT(*) AS first_matches, COUNT(points_last_5) AS form_not_null
    FROM team_form
    WHERE n_previous = 0
""")

,first_matches,form_not_null
0,224,0


In [4]:
query("""
    SELECT COUNT(*) AS team_rows,
           ROUND(AVG((f.n_previous >= 10)::INTEGER), 3) AS share_10_previous
    FROM team_form AS f
    JOIN clean_results AS c USING (match_id)
    WHERE c.date >= '2000-01-01'
""")

,team_rows,share_10_previous
0,47920,0.999


A team's first match has no form (NULL), as expected: nothing is known before it.
Since 2000, 99.9% of team rows have at least 10 previous matches, so the form features are almost always computed on a full window.
128 times a team played two matches on the same day (split squads in friendlies). The second match sees the first result. This is marginal and I note it as a limitation.

## 2. Elo ratings
I compute Elo ratings match by match on the full history, with the World Football Elo Ratings parameters: +100 home advantage only when the venue is not neutral, K from 20 (friendly) to 60 (World Cup), and a goal-difference multiplier.
Ratings are stored before each match is played, so they never include its result.

In [6]:
query("""
    WITH team_elo AS (
        SELECT c.date, c.tournament, c.home_team AS team, e.elo_home AS elo
        FROM clean_results AS c JOIN elo_ratings AS e USING (match_id)
        UNION ALL
        SELECT c.date, c.tournament, c.away_team, e.elo_away
        FROM clean_results AS c JOIN elo_ratings AS e USING (match_id)
    )
    SELECT team, ROUND(elo) AS elo_before_world_cup
    FROM team_elo
    WHERE tournament = 'FIFA World Cup' AND date >= '2026-06-01'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY team ORDER BY date) = 1
    ORDER BY elo DESC
    LIMIT 10
""")

,team,elo_before_world_cup
0,Spain,2193.0
1,Argentina,2168.0
2,France,2115.0
3,England,2067.0
4,Colombia,2064.0
5,Brazil,2058.0
6,Portugal,2029.0
7,Ecuador,2017.0
8,Netherlands,2008.0
9,Germany,2008.0


In [7]:
query("""
    SELECT c.neutral,
           ROUND(AVG(e.elo_home - e.elo_away), 1) AS mean_elo_gap,
           ROUND(AVG((e.elo_home > e.elo_away)::INTEGER), 3) AS share_home_stronger
    FROM elo_ratings AS e
    JOIN clean_results AS c USING (match_id)
    WHERE c.date >= '2000-01-01'
    GROUP BY c.neutral
    ORDER BY c.neutral
""")

,neutral,mean_elo_gap,share_home_stronger
0,False,20.8,0.537
1,True,20.9,0.555


In [8]:
query("""
    SELECT c.neutral,
           ROUND(AVG(e.elo_expected_home), 3) AS expected_score,
           ROUND(AVG(CASE c.result WHEN 'H' THEN 1 WHEN 'D' THEN 0.5 ELSE 0 END), 3) AS actual_score
    FROM elo_ratings AS e
    JOIN clean_results AS c USING (match_id)
    WHERE c.date >= '2000-01-01'
    GROUP BY c.neutral
    ORDER BY c.neutral
""")

,neutral,expected_score,actual_score
0,False,0.623,0.623
1,True,0.527,0.534


Before the 2026 World Cup, Spain, Argentina and France have the highest ratings, in line with reference rankings.
On neutral ground the listed home team is the stronger side 56% of the time (+21 Elo on average).
The average Elo expected score matches the actual score (0.623 vs 0.623 at home, 0.527 vs 0.534 on neutral ground). The listed home team advantage on neutral ground is explained by team strength, and the +100 home bonus fits the data.

## 3. Opponent strength
Recent goals mean little without the level of the opponents. I add the average pre-match Elo of the 10 previous opponents.

In [9]:
query("""
    SELECT m.team,
           ROUND(f.goals_for_last_10, 1) AS goals_for_last_10,
           ROUND(f.goals_against_last_10, 1) AS goals_against_last_10,
           ROUND(s.opponent_elo_last_10) AS opponent_elo_last_10
    FROM team_matches AS m
    JOIN team_form AS f USING (match_id, team)
    JOIN team_opponent_strength AS s USING (match_id, team)
    JOIN clean_results AS c USING (match_id)
    WHERE c.tournament = 'FIFA World Cup' AND c.date >= '2026-06-01'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY m.team ORDER BY m.date) = 1
    ORDER BY goals_for_last_10 DESC
    LIMIT 12
""")

,team,goals_for_last_10,goals_against_last_10,opponent_elo_last_10
0,Belgium,3.7,0.6,1579.0
1,Norway,3.1,0.8,1743.0
2,Germany,2.8,0.8,1670.0
3,Spain,2.8,0.4,1734.0
4,Austria,2.7,0.4,1465.0
5,Portugal,2.6,0.9,1709.0
6,Brazil,2.5,1.1,1867.0
7,Argentina,2.5,0.3,1683.0
8,France,2.5,0.9,1727.0
9,Turkey,2.5,1.3,1781.0


Belgium scored the most goals before the World Cup (3.7 per match), but against the weakest opponents (average Elo 1579), including two wins against Liechtenstein (Elo about 960). Brazil and Colombia scored 2.5 per match against much stronger sides (about 1865).
Goal averages need the opponent level to be meaningful, so both features go to the model together.